# In silico perturbations

Predict how knockouts and overexpression reshape the attractor landscape.

> **Starter notebook.** The cells below are a scaffold: real function calls with placeholder paths/arguments and `TODO` markers. Fill in your own data and run top-to-bottom. Anything marked `TODO` is a choice you need to make for your dataset.

## Setup and imports

In [ ]:
import os
import os.path as op
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import bobaT as bb

sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 100
import warnings
warnings.filterwarnings('ignore')

## Configure paths

In [ ]:
# Input data
DATA_DIR = './test_data'

# Output directories
OUTPUT_DIR = './output'
VAL_DIR = './output/validation'
ATTRACTOR_DIR = './output/attractors'
PERTURB_DIR = './output/perturbations'
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(PERTURB_DIR, exist_ok=True)

## Load network, rules, and attractors

In [ ]:
# Load the base network as a graph-tool graph
network_path = 'tf-lit-network.csv'  # TODO: your base network CSV
graph, vertex_dict = bb.load.load_network(
    f'{DATA_DIR}/{network_path}',
    remove_sinks=False, remove_selfloops=True, remove_sources=False,
)
v_names, nodes = bb.utils.get_nodes(vertex_dict, graph)

rules, regulators_dict = bb.load.load_rules(fname=f'{OUTPUT_DIR}/rules.txt')
attractor_dict = bb.utils.get_attractor_dict(ATTRACTOR_DIR, filtered=True)

## Run perturbation random walks

Set `perturbations=True` so each node is knocked out / forced on in turn.

In [ ]:
bb.rw.random_walks_parallel(
    attractor_dict, rules, regulators_dict, nodes,
    save_dir=PERTURB_DIR,
    radius=2, perturbations=True, iters=500, max_steps=500,
    stability=False, reach_or_leave='leave', random_start=False,
    on_nodes=[], off_nodes=[], overwrite_walks=False,
)

## Summarize perturbation effects

In [ ]:
bb.tl.perturbations_summary(
    attractor_dict, PERTURB_DIR, show=False, save=True, plot_by_attractor=True,
    save_dir='clustered_perturb_plots', save_full=True, significance='both',
    fname='', ncols=5, mean_threshold=-0.3,
)

## Per-gene and destabilization plots

In [ ]:
# Build the perturbation dictionary, then plot per-gene effects
p_dict = bb.utils.get_perturbation_dict(attractor_dict, PERTURB_DIR,
                                        significance='both', mean_threshold=-0.3)
# bb.plot.plot_perturb_gene_dictionary(p_dict, full, PERTURB_DIR)  # TODO: pass 'full' table

bb.plot.plot_destabilization_scores(attractor_dict, PERTURB_DIR, show=False, save=True)

## Related

See [random walks and trajectories](trajectories.ipynb) for unperturbed dynamics.